# 02 · 业务指标与多维分析 —— 先提问，再回答

**项目**：英国在线零售商经营诊断分析

> **方法论**：分析不是"把所有图都画一遍"，而是**先列出业务问题清单，再用数据逐一回答**。每个问题遵循同一个闭环：
>
> **业务问题 → 指标定义 → 计算 → 数据发现 → 业务解读**
>
> 这份问题清单就是本项目的"灵魂"，也是面试时讲述项目的骨架。

## 业务问题清单

| # | 业务问题 | 核心指标 | 分析方法 |
| --- | --- | --- | --- |
| Q1 | 生意在变好还是变坏？ | 月度 GMV / 订单数 / 客单价 | 时间趋势 |
| Q2 | 谁是最有价值的客户？如何分层运营？ | RFM | 客户分层 |
| Q3 | 拉来的新客能留住吗？ | 同期群月度留存 | Cohort 分析 |
| Q4 | 什么商品在支撑生意？ | 商品 GMV / 销量 / 集中度 | 帕累托分析 |
| Q5 | 钱从哪些市场来？ | 国家 GMV / 占比 | 结构分析 |
| Q6 | 客户什么时候下单？ | 星期 × 小时订单热力 | 行为规律 |
| Q7 | 退货吃掉了多少生意？ | 退货率 / 退货金额 | 专题诊断 |

## 0. 环境准备与数据加载

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 50)

# 只读清洗后的干净数据（01 已完成清洗），秒级加载
sales = pd.read_csv('../data/cleaned/sales_clean.csv', parse_dates=['InvoiceDate'])
returns = pd.read_csv('../data/cleaned/returns_clean.csv', parse_dates=['InvoiceDate'])
print(f'销售明细 {len(sales):,} 行 | 退货明细 {len(returns):,} 行')
sales.head(3)

销售明细 390,545 行 | 退货明细 10,617 行


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,InvoiceMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12


## Q1 · 经营概况：生意在变好还是变坏？

> **指标定义**（面试必背）：
> - **GMV** = Σ(数量 × 单价)：一段时期内的总成交额
> - **订单数** = 发票号去重计数：一张 Invoice = 一次交易
> - **活跃客户数** = 当月有购买的客户去重数
> - **客单价 AOV** = GMV / 订单数；**客户均消费 ARPPU** = GMV / 活跃客户数
>
> GMV 涨不一定健康：可能是"买的人多了"（拉新成功），也可能是"老客被薅狠了"（AOV 虚高）。所以要**四个指标放在一起看**，互相印证。

In [2]:
monthly = (sales.groupby('InvoiceMonth')
           .agg(GMV=('Revenue', 'sum'),
                订单数=('InvoiceNo', 'nunique'),
                活跃客户数=('CustomerID', 'nunique'))
           .reset_index())
monthly['客单价AOV'] = monthly['GMV'] / monthly['订单数']
monthly['客户均消费ARPPU'] = monthly['GMV'] / monthly['活跃客户数']
monthly['GMV环比'] = monthly['GMV'].pct_change()
monthly

,InvoiceMonth,GMV,订单数,活跃客户数,客单价AOV,客户均消费ARPPU,GMV环比
0,2010-12,"565,199.56",1394,884,405.45,639.37,NaN
1,2011-01,"562,682.91",983,739,572.41,761.41,-0.00
2,2011-02,"442,293.59",992,757,445.86,584.27,-0.21
3,2011-03,"583,143.85",1312,973,444.47,599.33,0.32
4,2011-04,"454,440.88",1139,853,398.98,532.76,-0.22
5,2011-05,"659,242.49",1544,1054,426.97,625.47,0.45
6,2011-06,"653,264.92",1390,990,469.97,659.86,-0.01
7,2011-07,"591,603.79",1321,946,447.85,625.37,-0.09
8,2011-08,"635,514.38",1267,933,501.59,681.15,0.07
9,2011-09,"938,752.63",1739,1259,539.82,745.63,0.48


In [3]:
total_gmv = sales['Revenue'].sum()
print(f'2011 年总 GMV（2010-12 起月均参考）：£{total_gmv:,.0f}')
print(f'总订单数：{sales["InvoiceNo"].nunique():,} | 总客户数：{sales["CustomerID"].nunique():,}')
print(f'整体客单价：£{total_gmv / sales["InvoiceNo"].nunique():,.0f}')
print(f'GMV 前 3 个月：{monthly.nlargest(3, "GMV")["InvoiceMonth"].tolist()}')
print(f'GMV 后 3 个月：{monthly.nsmallest(3, "GMV")["InvoiceMonth"].tolist()}')

2011 年总 GMV（2010-12 起月均参考）：£8,553,057
总订单数：18,361 | 总客户数：4,333
整体客单价：£466
GMV 前 3 个月：['2011-11', '2011-10', '2011-09']
GMV 后 3 个月：['2011-12', '2011-02', '2011-04']


### Q1 发现

- 月度 GMV 与订单量同向波动，**2011 年 9-11 月是全年高峰**（圣诞备货季），符合礼品零售的季节性规律；2 月、4 月是全年低谷；
- 客单价 AOV 全年围绕 **£466 的均值**在 £399~572 窄幅波动、**无明显上升或下降趋势**；
- 这说明 GMV 的涨跌主要来自"客户数/订单数"的变化，而不是单笔金额变化——**增长动能要看获客与留存**（引出 Q2、Q3）。

## Q2 · 客户价值：谁是最有价值的客户？（RFM 分层）

> **RFM 三件套**：
> - **R（Recency）最近一次购买距今天数**：越近越活跃
> - **F（Frequency）统计期内购买次数**（发票数）：越多次越忠诚
> - **M（Monetary）统计期内累计消费金额**：越大价值越高
>
> **打分方法**：各维度按**四分位**打 1~4 分。为什么用分位数而不是拍脑袋的阈值？——因为这是**数据驱动的相对分层**：这家店 F 的中位数是 2 次，另一家可能是 20 次，统一阈值没有意义。
>
> **8 分层**：R/F/M 各分高（≥3 分）低 → 2×2×2=8 类，每类对应一套运营动作。

In [4]:
# RF 快照日 = 数据最后一天 + 1，这样"最后一次购买当天"的 R = 1 天
snapshot = sales['InvoiceDate'].max() + pd.Timedelta(days=1)
print('RFM 快照日:', snapshot.date())

rfm = sales.groupby('CustomerID').agg(
    R=('InvoiceDate', lambda x: (snapshot - x.max()).days),
    F=('InvoiceNo', 'nunique'),
    M=('Revenue', 'sum'),
)
print(f'参与分层的客户：{len(rfm):,} 人')
rfm[['R', 'F', 'M']].describe()

RFM 快照日: 2011-12-09


参与分层的客户：4,333 人


,R,F,M
count,"4,333.00","4,333.00","4,333.00"
mean,92.46,4.24,"1,973.93"
std,100.07,7.62,"8,531.12"
min,1.00,1.00,2.90
25%,17.00,1.00,303.97
50%,51.00,2.00,660.00
75%,142.00,5.00,"1,624.05"
max,373.00,205.00,"279,138.02"


In [5]:
# 关键坑：F 分布严重右偏（一半人只买过 1~2 次），qcut 直接切会在重复值上报错
# 解决：先 rank(method='first') 打散并列值再分箱——每个客户都拿到唯一名次
rfm['R_score'] = pd.qcut(rfm['R'].rank(method='first'), 4, labels=[4, 3, 2, 1]).astype(int)  # R 越小分越高
rfm['F_score'] = pd.qcut(rfm['F'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_score'] = pd.qcut(rfm['M'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm[['R', 'F', 'M', 'R_score', 'F_score', 'M_score']].head()

,R,F,M,R_score,F_score,M_score
CustomerID,,,,,,
12346,325,1,"77,183.60",1,1,4
12347,2,7,"4,310.00",4,4,4
12348,75,4,"1,437.24",2,3,3
12349,18,1,"1,457.55",3,1,3
12350,310,1,294.40,1,1,1


In [6]:
def segment(r, f, m):
    """R/F/M 三维高中低 → 8 类客户（R>=3 记为"高"）"""
    if r >= 3 and f >= 3 and m >= 3: return '重要价值客户'
    if r >= 3 and f < 3 and m >= 3: return '重要发展客户'
    if r < 3 and f >= 3 and m >= 3: return '重要保持客户'
    if r < 3 and f < 3 and m >= 3: return '重要挽留客户'
    if r >= 3 and f >= 3: return '一般价值客户'
    if r >= 3: return '一般发展客户'
    if f >= 3: return '一般保持客户'
    return '一般挽留客户'

rfm['客户分层'] = [segment(r, f, m) for r, f, m in zip(rfm['R_score'], rfm['F_score'], rfm['M_score'])]

seg_summary = (rfm.groupby('客户分层')
               .agg(客户数=('M', 'size'), GMV=('M', 'sum'), 平均F=('F', 'mean'))
               .sort_values('GMV', ascending=False))
seg_summary['客户数占比'] = seg_summary['客户数'] / seg_summary['客户数'].sum()
seg_summary['GMV占比'] = seg_summary['GMV'] / seg_summary['GMV'].sum()
seg_summary

,客户数,GMV,平均F,客户数占比,GMV占比
客户分层,,,,,
重要价值客户,1311,"6,355,077.63",9.32,0.30,0.74
重要保持客户,454,"935,181.11",4.66,0.10,0.11
重要挽留客户,268,"419,338.32",1.54,0.06,0.05
一般挽留客户,1249,"350,724.78",1.18,0.29,0.04
一般发展客户,517,"159,982.21",1.34,0.12,0.02
重要发展客户,133,"159,167.96",1.83,0.03,0.02
一般价值客户,206,"93,359.52",3.01,0.05,0.01
一般保持客户,195,"80,225.45",2.99,0.05,0.01


### Q2 发现

- **强二八分化**："重要价值客户"以 **30% 的人数贡献了 74% 的 GMV**；而人数占 29% 的"一般挽留客户"只贡献 4%——典型的批发客户结构；
- **重要挽留客户**（268 人，消费高但最近沉默）是最值得花钱召回的人群：他们有钱有意愿，只是流失了；
- **重要发展客户**（133 人，最近活跃但频次低）适合用组合装/满减提升频次。

## Q3 · 用户生命周期：拉来的新客，下个月还在吗？（Cohort 留存）

> **同期群（Cohort）分析**：把客户按"首次购买月"分组，追踪每个群在之后第 1、2、3…个月的**复访率**。
>
> 为什么不用"月度复购率"一个数字？——因为它把"老客越来越多"和"留存变差"混在一起；同期群把**获客质量随时间的变化**拆开看，是国际通行的留存分析标准做法。

In [7]:
first_month = sales.groupby('CustomerID')['InvoiceMonth'].min().rename('首次购买月')
co = sales.merge(first_month, on='CustomerID')

# 期数 = 当前购买月与首次购买月的月差（0 = 首月）
co['月份'] = pd.to_datetime(co['InvoiceMonth'] + '-01')
co['首次月'] = pd.to_datetime(co['首次购买月'] + '-01')
co['期数'] = (co['月份'].dt.year - co['首次月'].dt.year) * 12 + (co['月份'].dt.month - co['首次月'].dt.month)

retention = co.pivot_table(index='首次购买月', columns='期数',
                           values='CustomerID', aggfunc='nunique')
cohort_size = retention.iloc[:, 0]
retention_pct = retention.divide(cohort_size, axis=0).round(3)
retention_pct

期数,0,1,2,3,4,5,6,7,8,9,10,11,12
首次购买月,,,,,,,,,,,,,
2010-12,1.00,0.36,0.32,0.38,0.36,0.40,0.36,0.35,0.35,0.40,0.37,0.50,0.25
2011-01,1.00,0.22,0.27,0.23,0.32,0.29,0.25,0.24,0.30,0.33,0.36,0.12,NaN
2011-02,1.00,0.19,0.19,0.29,0.27,0.24,0.26,0.27,0.25,0.30,0.07,NaN,NaN
2011-03,1.00,0.15,0.25,0.20,0.22,0.17,0.27,0.23,0.28,0.08,NaN,NaN,NaN
2011-04,1.00,0.21,0.20,0.21,0.20,0.23,0.22,0.26,0.07,NaN,NaN,NaN,NaN
2011-05,1.00,0.19,0.17,0.17,0.21,0.23,0.26,0.09,NaN,NaN,NaN,NaN,NaN
2011-06,1.00,0.17,0.16,0.26,0.23,0.33,0.10,NaN,NaN,NaN,NaN,NaN,NaN
2011-07,1.00,0.18,0.20,0.23,0.27,0.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011-08,1.00,0.20,0.24,0.24,0.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
print(f'各月新客规模（cohort 人数）：')
print(cohort_size.to_string())
m1 = retention_pct[1].dropna().mean()
m3 = retention_pct[3].dropna().mean() if 3 in retention_pct.columns else float('nan')
print(f'\n平均次月留存：{m1:.1%} | 平均 3 月留存：{m3:.1%}')

各月新客规模（cohort 人数）：
首次购买月
2010-12   884.00
2011-01   416.00
2011-02   380.00
2011-03   452.00
2011-04   300.00
2011-05   284.00
2011-06   242.00
2011-07   187.00
2011-08   169.00
2011-09   299.00
2011-10   357.00
2011-11   323.00
2011-12    40.00

平均次月留存：20.4% | 平均 3 月留存：23.2%


### Q3 发现

- **次月留存极低（约两成）**：超过 3/4 的客户买一次就不再回来——这是这家店**最大的漏水口**；
- 2010-12 首月群（圣诞季获客）的长期留存略好于其他月份，说明**节日获客质量不差，差的是节日之后的承接**；
- 结论方向：问题不在"拉新"，在"首购后的第二次转化"（催复购动作缺位）。

## Q4 · 商品分析：什么商品在支撑生意？

> **帕累托（二八）检验**：把商品按 GMV 从大到小排，看"累计 GMV 占比"曲线——多少比例的商品贡献了 80% 的 GMV？
> 如果约 20% 的商品贡献 80% 收入，说明经营高度依赖头部商品：**头部断货 = 业务事故**，尾部则要考虑清理 SKU。

In [9]:
# 每个商品取最常见的描述作为展示名（同一 StockCode 描述可能不统一）
name_map = sales.groupby('StockCode')['Description'].agg(lambda x: x.mode().iat[0] if len(x.mode()) else '')

prod = (sales.groupby('StockCode')
        .agg(GMV=('Revenue', 'sum'), 销量=('Quantity', 'sum'))
        .sort_values('GMV', ascending=False))
prod['商品名'] = name_map
prod.head(10)

,GMV,销量,商品名
StockCode,,,
22423,"142,137.25",12364,REGENCY CAKESTAND 3 TIER
85123A,"100,544.50",36762,WHITE HANGING HEART T-LIGHT HOLDER
85099B,"85,019.74",46068,JUMBO BAG RED RETROSPOT
23166,"81,366.81",77868,MEDIUM CERAMIC TOP STORAGE JAR
47566,"68,785.23",15279,PARTY BUNTING
84879,"56,385.99",35247,ASSORTED COLOUR BIRD ORNAMENT
23084,"51,098.76",27073,RABBIT NIGHT LIGHT
22502,"47,348.35",1485,PICNIC BASKET WICKER SMALL
79321,"46,253.61",9644,CHILLI LIGHTS


In [10]:
# 帕累托：累计 GMV 占比
prod_sorted = prod.sort_values('GMV', ascending=False).reset_index()
prod_sorted['累计GMV占比'] = prod_sorted['GMV'].cumsum() / prod_sorted['GMV'].sum()
n80 = (prod_sorted['累计GMV占比'] < 0.8).sum() + 1
print(f'商品总数：{len(prod_sorted):,} 个')
print(f'贡献 80% GMV 只需前 {n80:,} 个商品（占 {n80 / len(prod_sorted):.1%}）')
print(f'前 10 名商品 GMV 占比：{prod_sorted["GMV"].head(10).sum() / total_gmv:.1%}')

商品总数：3,658 个
贡献 80% GMV 只需前 796 个商品（占 21.8%）
前 10 名商品 GMV 占比：8.4%


### Q4 发现

- 头部商品以**手工风节庆礼品**为主：蛋糕三层架（REGENCY CAKESTAND，£14.2 万居首）、心形烛台挂饰、复古购物袋、陶瓷储物罐等，单价多在 £3~12，**GMV 靠走量**；
- 帕累托近似成立：**21.8% 的商品（796 个）贡献 80% GMV**，头部集中度高；
- 销量榜与 GMV 榜不完全重合：低单价高销量的"引流品"与高单价"利润品"要分开运营。

## Q5 · 市场结构：钱从哪些国家来？

> 看两个口径：① 英国本土 vs 海外的 GMV 占比；② 海外市场内部的排名。海外市场多为欧洲邻国，物流近、文化近，是自然扩张方向。

In [11]:
country = (sales.groupby('Country')
           .agg(GMV=('Revenue', 'sum'), 订单数=('InvoiceNo', 'nunique'), 客户数=('CustomerID', 'nunique'))
           .sort_values('GMV', ascending=False))
uk_gmv = country.loc['United Kingdom', 'GMV'] if 'United Kingdom' in country.index else country.iloc[0]['GMV']
print(f'英国本土 GMV 占比：{uk_gmv / total_gmv:.1%}')
country.head(10)

英国本土 GMV 占比：82.6%


,GMV,订单数,客户数
Country,,,
United Kingdom,"7,063,312.16",16546,3916
Netherlands,"283,889.34",93,9
EIRE,"257,013.12",256,3
Germany,"203,835.43",439,93
France,"183,552.49",378,87
Australia,"138,103.81",56,9
Spain,"55,706.56",88,30
Switzerland,"52,441.95",47,21
Japan,"37,416.37",19,8


In [12]:
overseas = country.drop(index='United Kingdom', errors='ignore').head(9)
overseas[['GMV', '订单数', '客户数']]

,GMV,订单数,客户数
Country,,,
Netherlands,"283,889.34",93,9
EIRE,"257,013.12",256,3
Germany,"203,835.43",439,93
France,"183,552.49",378,87
Australia,"138,103.81",56,9
Spain,"55,706.56",88,30
Switzerland,"52,441.95",47,21
Japan,"37,416.37",19,8
Sweden,"36,828.83",34,8


### Q5 发现

- 这是一家**以英国本土为绝对主体**的生意（82.6% GMV），客户是批发商；
- 海外市场典型的"**少数大批发客**"结构：**荷兰仅 9 个客户贡献 £28.4 万、爱尔兰 3 个客户贡献 £25.7 万**——单客价值极高，但大客户流失/转向的风险也高度集中；
- 德国、法国客户数较多（93/87 人）且客单结构分散，是更"健康"的可深耕市场；澳洲等远市场单均金额大，**物流成本与退货风险要单独评估**（呼应 Q7）。

## Q6 · 行为规律：客户在什么时间下单？

> **星期 × 小时** 的订单热力表，回答"投放、直播、秒杀应该排什么时间"。

In [13]:
sales['小时'] = sales['InvoiceDate'].dt.hour
sales['星期'] = sales['InvoiceDate'].dt.dayofweek          # 0=周一 … 6=周日
sales['星期名'] = sales['星期'].map({0: '周一', 1: '周二', 2: '周三', 3: '周四', 4: '周五', 5: '周六', 6: '周日'})

hw = sales.pivot_table(index='星期', columns='小时', values='InvoiceNo', aggfunc='nunique')
hw = hw.reindex(range(7)).reindex(columns=range(7, 21))    # 补齐缺失行/列（周六全天空缺是数据真实特征）
hw.index = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
hw

小时,7,8,9,10,11,12,13,14,15,16,17,18,19,20
周一,4.00,85.00,250.00,319.00,334.00,478.00,433.00,366.00,314.00,179.00,68.00,NaN,NaN,NaN
周二,5.00,114.00,286.00,388.00,403.00,528.00,473.00,371.00,340.00,159.00,86.00,3.00,NaN,NaN
周三,5.00,116.00,288.00,429.00,409.00,609.00,492.00,415.00,369.00,222.00,79.00,NaN,2.00,1.00
周四,9.00,119.00,292.00,438.00,368.00,583.00,475.00,438.00,418.00,307.00,244.00,164.00,132.00,16.00
周五,6.00,116.00,255.00,397.00,357.00,448.00,395.00,355.00,250.00,115.00,62.00,2.00,9.00,1.00
周六,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
周日,NaN,NaN,2.00,225.00,385.00,459.00,355.00,306.00,330.00,106.00,NaN,NaN,NaN,NaN


In [14]:
top_slots = sales.groupby(['星期', '小时'])['InvoiceNo'].nunique().nlargest(5)
print('订单量最高的 5 个时段：')
for (w, h), v in top_slots.items():
    print(f'  星期{"一二三四五六日"[w]} {h:02d}:00 — {v:,} 单')
sat_orders = sales.loc[sales['星期'] == 5, 'InvoiceNo'].nunique()
print(f'周六订单总数：{sat_orders:,}（该公司周六不发货，属真实业务特征）')

订单量最高的 5 个时段：
  星期三 12:00 — 609 单
  星期四 12:00 — 583 单
  星期二 12:00 — 528 单
  星期三 13:00 — 492 单
  星期一 12:00 — 478 单
周六订单总数：0（该公司周六不发货，属真实业务特征）


### Q6 发现

- 订单集中在**工作日 10:00~15:00**（B2B 批发客户上班时间采购），中午 12 点附近最旺；
- **周六全天空缺**：真实业务特征（批发商周六不采购）；
- 落地动作：营销推送、客服排班、促销活动都应对齐**工作日上午到午后**的窗口，周末投放基本是浪费。

## Q7 · 退货诊断：退货吃掉了多少生意？

> **口径定义**（面试常被追问，也是本项目踩过的真实的坑）：
> - 退货表：C 开头发票或数量为负的行；
> - **退货金额** = |数量| × 单价（退货行单价为正、数量为负）；
> - **关键发现**：退货表里混着大量**非商品冲销行**——AMAZONFEE（亚马逊平台费冲销）、M（手工调整）、POST（运费）、BANK CHARGES 等。它们不是"客户退货"，而是财务调整！
> - 因此把口径拆成两个：**整体口径**（含财务调整，反映财务流水）与**商品口径**（只算真实商品退货，反映商品/物流质量）。商品分析一律用商品口径。

In [15]:
returns['ReturnAmount'] = -returns['Quantity'] * returns['UnitPrice']
special_codes = {'POST', 'DOT', 'M', 'D', 'CRUK', 'C2', 'PADS', 'S',
                 'BANK CHARGES', 'ADJUST', 'AMAZONFEE', 'GIFT'}
sc = returns['StockCode'].astype(str).str.upper()
is_product_ret = (~sc.isin(special_codes)) & (~sc.str.lower().str.startswith('gift'))
prod_returns = returns[is_product_ret].copy()

total_ret_amt = returns['ReturnAmount'].sum()
prod_ret_amt = prod_returns['ReturnAmount'].sum()
nonprod_amt = total_ret_amt - prod_ret_amt
print(f'退货流水合计：£{total_ret_amt:,.0f}')
print(f'  其中非商品冲销（平台费/手工调整/运费等）：£{nonprod_amt:,.0f}（占 {nonprod_amt / total_ret_amt:.0%}）')
print(f'  真实商品退货金额：£{prod_ret_amt:,.0f}')
print(f'退货率（整体口径）：{total_ret_amt / (total_ret_amt + total_gmv):.2%}')
print(f'退货率（商品口径）：{prod_ret_amt / (prod_ret_amt + total_gmv):.2%}')
print(f'退货单数：{returns["InvoiceNo"].nunique():,} | 涉及商品：{prod_returns["StockCode"].nunique():,} 个')

退货流水合计：£728,023
  其中非商品冲销（平台费/手工调整/运费等）：£417,864（占 57%）
  真实商品退货金额：£310,160
退货率（整体口径）：7.84%
退货率（商品口径）：3.50%
退货单数：5,167 | 涉及商品：2,549 个


In [16]:
# 月度退货率用"商品口径"，避免平台费冲销的月度波动污染商品质量趋势
ret_month = (prod_returns.assign(月=prod_returns['InvoiceDate'].dt.to_period('M').astype(str))
             .groupby('月')['ReturnAmount'].sum())
ret_month_rate = ret_month / (ret_month + monthly.set_index('InvoiceMonth')['GMV'])
pd.DataFrame({'商品退货金额': ret_month, '月度退货率(商品口径)': ret_month_rate})

,商品退货金额,月度退货率(商品口径)
月,,
2010-12,"17,607.86",0.03
2011-01,"91,526.30",0.14
2011-02,"8,375.87",0.02
2011-03,"10,675.30",0.02
2011-04,"33,316.91",0.07
2011-05,"8,948.23",0.01
2011-06,"13,760.84",0.02
2011-07,"11,409.21",0.02
2011-08,"22,929.52",0.03


In [17]:
ret_prod = (prod_returns.groupby('StockCode')['ReturnAmount'].sum().nlargest(10).to_frame('退货金额'))
ret_prod['商品名'] = name_map.reindex(ret_prod.index)
ret_prod

,退货金额,商品名
StockCode,,
23166,"77,479.64",MEDIUM CERAMIC TOP STORAGE JAR
22423,"9,722.55",REGENCY CAKESTAND 3 TIER
85123A,"6,624.30",WHITE HANGING HEART T-LIGHT HOLDER
21108,"6,591.42",FAIRY CAKE FLANNEL ASSORTED COLOUR
23113,"4,803.06",PANTRY CHOPPING BOARD
48185,"4,554.90",DOORMAT FAIRY CAKE
21175,"3,775.33",GIN + TONIC DIET METAL SIGN
47566B,"3,692.95",TEA TIME PARTY BUNTING
22273,"3,512.65",FELTCRAFT DOLL MOLLY


### Q7 发现

- **最大的坑在这里**：按流水口径退货率高达约 8%，但拆开一看，**超过一半是亚马逊平台费冲销与手工调整**，真实商品退货率只有约 3~4%——不拆口径，"退货治理"的方向就会跑偏；
- 真实退货高度集中于**少数商品**（陶瓷储物罐等易碎大件），"黑名单商品"是质量改进的第一抓手；
- 月度波动没有明显恶化趋势，说明是**结构性问题（个别商品/个别客户）而非系统性恶化**。

## 分析结果落盘

In [18]:
# 把分析中间结果存盘，供 03 可视化与 README 引用
monthly.to_csv('../data/cleaned/analysis_monthly.csv', index=False)
rfm.reset_index().to_csv('../data/cleaned/analysis_rfm.csv', index=False)
retention_pct.to_csv('../data/cleaned/analysis_retention.csv')
country.reset_index().to_csv('../data/cleaned/analysis_country.csv', index=False)
hw.to_csv('../data/cleaned/analysis_hour_week.csv')
prod_sorted.to_csv('../data/cleaned/analysis_products.csv', index=False)
ret_month.to_csv('../data/cleaned/analysis_return_month.csv')
print('分析结果已保存至 data/cleaned/')

分析结果已保存至 data/cleaned/

## 本章小结：一张"诊断书"

| 业务问题 | 一句话诊断 |
| --- | --- |
| Q1 经营概况 | GMV 随季节波动，9-11 月为旺季；客单价稳定，增长靠客户数 |
| Q2 客户分层 | 二八分化显著，少数高价值客户撑起大盘 |
| Q3 留存 | 次月留存极低，"首购后的第二次转化"是最大漏水口 |
| Q4 商品 | 头部商品集中度高，库存与选品应向头部倾斜 |
| Q5 市场 | 英国本土为主，欧洲邻国是自然扩张方向 |
| Q6 时段 | 工作日 10-15 点是黄金窗口，周六停业 |
| Q7 退货 | 退货集中于少数商品，结构性问题而非系统性恶化 |

> 下一章（03）把这些发现全部**图形化**，并落成可执行的**业务建议**。